# Phase 2 — Proposed Improved Method: Confidence-Stabilized, Terrain-Cue Guidance

**Research direction (proposal, not yet validated):** combine a stronger person detector/tracker with temporal confidence smoothing and a transparent terrain-cue risk layer. The objective is to avoid changing guidance state on every single missed detection and to expose possible terrain cues in the output.

**Novelty must be demonstrated, not assumed.** YOLOv8 and SegFormer are existing models. The potentially novel contribution is the *decision policy and evaluation protocol*: temporal stabilization of target presence + a spatial scene-cue layer + explicit FOLLOW / SEARCH / CAUTION / STOP rules. In this notebook, ADE20K labels are only generic scene cues, not golf-specific hazard labels. Until annotated terrain ground truth is available, do not claim safe/unsafe classification or proven performance gains.

## 1. Install packages

In [ ]:
%pip -q install ultralytics transformers opencv-python-headless pandas matplotlib pillow scikit-learn

## 2. Upload the same video used for the baseline

In [ ]:
from google.colab import files
uploaded = files.upload()
video_files = [n for n in uploaded if n.lower().endswith(('.mp4','.mov','.avi','.mkv','.webm'))]
if not video_files: raise ValueError("Upload the SAME video used in the baseline.")
VIDEO_PATH = video_files[0]
print("Using:", VIDEO_PATH)

## 3. Sample the same 50 frames

In [ ]:
import cv2,time,math,os,zipfile,json
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from PIL import Image
cap=cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened(): raise RuntimeError("Could not open video.")
TOTAL=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); FPS=cap.get(cv2.CAP_PROP_FPS) or 0
indices=np.linspace(0,TOTAL-1,min(50,TOTAL),dtype=int)
frames=[]
for idx in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(idx)); ok,bgr=cap.read()
    if ok: frames.append((int(idx),cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)))
cap.release()
print(f"Sampled {len(frames)} frames from {TOTAL}; {FPS:.2f} FPS")

## 4. YOLOv8 person detection + tracker identity

In [ ]:
from ultralytics import YOLO
yolo=YOLO("yolov8n.pt")
yolo_rows=[]; yolo_vis=[]
for i,(frame_no,rgb) in enumerate(frames):
    start=time.perf_counter()
    # Detection mode is used on spaced samples. Tracking IDs are not compared across
    # widely spaced frames here; this notebook smooths presence over the sampled sequence.
    result=yolo.predict(source=rgb,classes=[0],conf=0.15,imgsz=640,verbose=False)[0]
    elapsed=time.perf_counter()-start
    boxes=result.boxes
    n=0 if boxes is None else len(boxes)
    confs=[] if not n else boxes.conf.cpu().numpy().tolist()
    max_conf=float(max(confs)) if confs else 0.0
    annotated=cv2.cvtColor(result.plot(),cv2.COLOR_BGR2RGB)
    yolo_vis.append(annotated)
    yolo_rows.append({"sample_index":i,"frame_no":frame_no,"person_detected_raw":int(n>0),
                      "person_count":n,"max_confidence":max_conf,"yolo_seconds":elapsed})
yolo_df=pd.DataFrame(yolo_rows)
display(yolo_df.head())

## 5. Temporal smoothing of target presence

In [ ]:
# Policy parameters can be changed for ablation testing.
WINDOW = 3          # majority vote over current + recent sampled observations
LOST_TOLERANCE = 2  # tolerate up to this many consecutive missing observations
raw = yolo_df["person_detected_raw"].astype(int).tolist()
smoothed=[]
for i in range(len(raw)):
    lo=max(0,i-WINDOW+1)
    recent=raw[lo:i+1]
    smoothed.append(int(sum(recent) >= math.ceil(len(recent)/2)))
states=[]; lost_run=0
for present in smoothed:
    if present:
        lost_run=0; states.append("FOLLOW")
    else:
        lost_run+=1
        states.append("SEARCH" if lost_run <= LOST_TOLERANCE else "STOP")
yolo_df["person_detected_smoothed"]=smoothed
yolo_df["presence_state"]=states
display(yolo_df[["frame_no","person_detected_raw","person_detected_smoothed","presence_state"]].head(15))

## 6. SegFormer scene cues (not golf hazard ground truth)

In [ ]:
import torch
import torch.nn.functional as F
from transformers import SegformerImageProcessor,SegformerForSemanticSegmentation
device="cuda" if torch.cuda.is_available() else "cpu"
SEG_NAME="nvidia/segformer-b0-finetuned-ade-512-512"
processor=SegformerImageProcessor.from_pretrained(SEG_NAME)
seg_model=SegformerForSemanticSegmentation.from_pretrained(SEG_NAME).to(device).eval()
id2label=seg_model.config.id2label

def predict_mask(rgb):
    inp=processor(images=Image.fromarray(rgb),return_tensors="pt")
    inp={k:v.to(device) for k,v in inp.items()}
    with torch.no_grad(): logits=seg_model(**inp).logits
    logits=F.interpolate(logits,size=rgb.shape[:2],mode="bilinear",align_corners=False)
    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)

masks=[]; seg_rows=[]
for i,(frame_no,rgb) in enumerate(frames):
    start=time.perf_counter(); mask=predict_mask(rgb); sec=time.perf_counter()-start
    masks.append(mask)
    ids,counts=np.unique(mask,return_counts=True)
    percentages={id2label.get(int(k),str(k)):float(v/mask.size) for k,v in zip(ids,counts)}
    # Generic scene cue only. These labels are not a validated traversability map.
    generic_cue_pct=sum(percentages.get(name,0.0) for name in ["water","river","sea","lake"])
    seg_rows.append({"sample_index":i,"frame_no":frame_no,"segformer_seconds":sec,
                     "generic_water_scene_fraction":generic_cue_pct,
                     "top_scene_labels":"; ".join(f"{k}:{v*100:.1f}%" for k,v in sorted(percentages.items(),key=lambda x:x[1],reverse=True)[:5])})
seg_df=pd.DataFrame(seg_rows)
display(seg_df.head())

## 7. Transparent guidance policy and demo visualization

In [ ]:
# CAUTION is triggered only by a generic water-like scene cue in the lower half of the frame.
# This is an experimental proxy, not a golf-course hazard detector and not a safety guarantee.
WATER_CUE_THRESHOLD=0.12
guidance=[]
for i,(frame_no,rgb) in enumerate(frames):
    # Evaluate cue on lower half, where ground often appears in forward-facing footage.
    mask=masks[i]
    lower=mask[mask.shape[0]//2:,:]
    labels,counts=np.unique(lower,return_counts=True)
    pct={id2label.get(int(k),str(k)):float(v/lower.size) for k,v in zip(labels,counts)}
    water_fraction=sum(pct.get(name,0.0) for name in ["water","river","sea","lake"])
    presence=states[i]
    if presence=="STOP": state="STOP"
    elif presence=="SEARCH": state="SEARCH"
    elif water_fraction>=WATER_CUE_THRESHOLD: state="CAUTION"
    else: state="FOLLOW"
    guidance.append({"frame_no":frame_no,"state":state,"lower_frame_water_cue_fraction":water_fraction})
guidance_df=pd.DataFrame(guidance)
display(guidance_df.head(15))

def overlay_mask(rgb,mask):
    # Colourize labels for visualization; colour has no safety meaning.
    rng=np.random.default_rng(7)
    palette=rng.integers(0,255,size=(max(256,int(mask.max())+1),3),dtype=np.uint8)
    return cv2.addWeighted(rgb,0.62,palette[mask],0.38,0)

demo_i=min(25,len(frames)-1)
rgb=frames[demo_i][1]
overlay=overlay_mask(rgb,masks[demo_i])
cv2.putText(overlay,f"STATE: {guidance_df.iloc[demo_i]['state']}",(12,32),
            cv2.FONT_HERSHEY_SIMPLEX,.9,(255,255,255),2,cv2.LINE_AA)
fig,ax=plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(rgb); ax[0].set_title("Original")
ax[1].imshow(yolo_vis[demo_i]); ax[1].set_title("YOLOv8 person detections")
ax[2].imshow(overlay); ax[2].set_title("Segmentation overlay + proposed state")
for a in ax:a.axis("off")
plt.tight_layout();plt.show()

## 8. Export improved outputs

In [ ]:
OUT="/content/improved_method_outputs";os.makedirs(OUT,exist_ok=True)
combined=yolo_df.merge(seg_df,on=["sample_index","frame_no"]).merge(guidance_df,on="frame_no")
combined.to_csv(os.path.join(OUT,"improved_frame_results.csv"),index=False)
summary={"method":"YOLOv8n + temporal presence smoothing + SegFormer generic scene cue policy",
         "sampled_frames":len(frames),
         "raw_person_detection_rate_percent":round(100*float(yolo_df.person_detected_raw.mean()),2),
         "smoothed_presence_rate_percent":round(100*float(yolo_df.person_detected_smoothed.mean()),2),
         "mean_yolo_seconds":round(float(yolo_df.yolo_seconds.mean()),4),
         "mean_segformer_seconds":round(float(seg_df.segformer_seconds.mean()),4),
         "guidance_state_counts":guidance_df.state.value_counts().to_dict(),
         "ground_truth_metrics":"Not calculated; verified person labels and terrain masks are not supplied",
         "safety_note":"CAUTION cue is a generic ADE20K water label proxy, not validated golf hazard detection."}
with open(os.path.join(OUT,"improved_summary.json"),"w") as f:json.dump(summary,f,indent=2)
Image.fromarray(yolo_vis[demo_i]).save(os.path.join(OUT,"yolo_detection_demo.png"))
Image.fromarray(overlay).save(os.path.join(OUT,"proposed_guidance_demo.png"))
print(json.dumps(summary,indent=2))
zip_path="/content/Improved_Method_Outputs.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for name in os.listdir(OUT):z.write(os.path.join(OUT,name),arcname=name)
files.download(zip_path)

## 9. Ablation and fair comparison

Run both notebooks on the **same input video and same sampled frame indices**. Compare raw person-detection rate, latency, and state flicker (number of guidance-state changes). Do not claim improved detection accuracy unless manually verified ground-truth labels are available. Do not claim golf-terrain safety without golf-specific terrain/hazard labels and a validated evaluation.